# Experiment: Module 12 — Champion–challenger registry audit

**Question.** Which model is supported as the current champion, and does any existing challenger have evidence that can lawfully trigger promotion?

**Success criteria.** Every source report and embedded hash must validate; TF-IDF must remain the historical champion; no BANKING77 text or test rows may be loaded; existing development evidence must remain promotion-ineligible; and the final decision must be `retain_champion` while the external evaluation lock is missing.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (current, *current.parents) if (path / 'pyproject.toml').is_file())
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
'Repository-local audit environment initialised'

'Repository-local audit environment initialised'

## Audit plan

1. Parse the strict Module 12 configuration and verify every registered evidence file.
2. Validate the committed, self-hashing registry snapshot.
3. Compare historical single-seed context without treating it as new promotion evidence.
4. Inspect challenger lifecycle status and evidence eligibility.
5. Confirm the service-model mismatch, missing external lock and fail-closed decision.

In [2]:
from governed_banking.champion import ChampionChallengerConfig, validate_registry_report
from governed_banking.data import sha256_file

config = ChampionChallengerConfig.from_yaml(PROJECT_ROOT / 'configs/champion_challenger.yaml')
registry_path = PROJECT_ROOT / 'reports/champion/champion-registry.json'
registry = json.loads(registry_path.read_text(encoding='utf-8'))
validate_registry_report(registry, config=config)
assert registry['implementation_sha256'] == {
    'build_champion_registry.py': sha256_file(PROJECT_ROOT / 'scripts/build_champion_registry.py'),
    'champion.py': sha256_file(PROJECT_ROOT / 'src/governed_banking/champion.py'),
}
{
    'framework': registry['framework_version'],
    'champion': registry['current_champion_id'],
    'decision': registry['current_decision']['action'],
    'registry_sha256': registry['report_sha256'],
}

{'framework': 'module12-champion-challenger-v1',
 'champion': 'tfidf-word-char-c4',
 'decision': 'retain_champion',
 'registry_sha256': '13a5299cedf4a33a2bc6911c932a2289d7960acf4b9372925616b3aa10ca7d3c'}

## Historical context—not a new model comparison

The table below reproduces already-observed BANKING77 test metrics from hash-bound reports. It explains the current champion but cannot be reused to promote a newly tuned model.

In [3]:
import pandas as pd

historical = pd.DataFrame(registry['historical_ranking_context_only'])
historical[['model_id', 'macro_f1', 'promotion_eligible']]

,model_id,macro_f1,promotion_eligible
0,tfidf-word-char-c4,0.905301,False
1,frozen-roberta-mean-c1024,0.896417,False
2,lora-roberta-r8-original,0.820206,False


## Challenger readiness

A model may be technically promising and still be ineligible for promotion. This distinction prevents validation reuse, calibration-only arguments or model fashion from overriding the evaluation boundary.

In [4]:
readiness = pd.DataFrame([
    {
        'model_id': model['model_id'],
        'role': model['role'],
        'status': model['lifecycle_status'],
        'evidence_items': len(model['evidence']),
        'external_locked_evaluation': model['external_locked_evaluation_present'],
        'promotion_ready': model['promotion_ready'],
    }
    for model in registry['models']
])
readiness

,model_id,role,status,evidence_items,external_locked_evaluation,promotion_ready
0,tfidf-word-char-c4,champion,active_historical_champion,1,False,False
1,frozen-roberta-mean-c1024,challenger,historical_challenger,1,False,False
2,lora-roberta-r8-original,challenger,retired_historical_challenger,1,False,False
3,lora-roberta-r8-revised,challenger,active_development_challenger,3,False,False
4,full-roberta-base,challenger,planned_cuda_challenger,0,False,False


In [5]:
revised_lora = next(model for model in registry['models'] if model['model_id'] == 'lora-roberta-r8-revised')
{
    'development_mean_macro_f1': revised_lora['development_validation_macro_f1']['mean'],
    'development_calibrated_ece': revised_lora['development_calibrated_ece']['mean'],
    'calibration_point_gates_passed': revised_lora['calibration_point_gates_passed'],
    'development_selective_risk': revised_lora['development_selective_risk']['mean'],
    'development_possible_ood_recall': revised_lora['development_possible_ood_recall']['mean'],
    'uncertainty_gates_passed': revised_lora['uncertainty_gates_passed'],
    'promotion_ready': revised_lora['promotion_ready'],
}

{'development_mean_macro_f1': 0.8974052122,
 'development_calibrated_ece': 0.0280082064,
 'calibration_point_gates_passed': True,
 'development_selective_risk': 0.064345403,
 'development_possible_ood_recall': 0.8888888889,
 'uncertainty_gates_passed': False,
 'promotion_ready': False}

## Registered promotion paths

Route A requires macro-F1 superiority for all three seeds with a positive paired confidence-interval lower bound. Route B requires macro-F1 non-inferiority plus a statistically supported material calibration or selective-risk improvement. Both routes remain subject to security-intent, privacy, routing and audit vetoes, followed by human approval.

In [6]:
gates = config.promotion_gates
{
    'required_seeds': gates['required_seeds'],
    'superiority_ci_lower_above': gates['superiority']['mean_macro_f1_delta_ci_lower_strictly_above'],
    'noninferiority_margin': gates['noninferiority']['mean_macro_f1_delta_ci_lower_at_least'],
    'minimum_mean_ece_reduction': gates['calibration_route']['minimum_mean_ece_reduction'],
    'minimum_mean_selective_risk_reduction': gates['selective_risk_route']['minimum_mean_selective_risk_reduction'],
    'automatic_promotion_permitted': gates['approval']['automatic_promotion_permitted'],
    'human_approval_required': gates['approval']['human_approval_required'],
}

{'required_seeds': [17, 42, 73],
 'superiority_ci_lower_above': 0.0,
 'noninferiority_margin': -0.005,
 'minimum_mean_ece_reduction': 0.01,
 'minimum_mean_selective_risk_reduction': 0.02,
 'automatic_promotion_permitted': False,
 'human_approval_required': True}

In [7]:
assert registry['data_boundary'] == {
    'source_reports_only': True,
    'message_text_loaded': False,
    'official_banking77_test_loaded': False,
    'external_evaluation_loaded': False,
    'new_model_metrics_computed': False,
}
assert registry['promotion_readiness']['external_evaluation_lock_status'] == 'missing'
assert registry['promotion_readiness']['eligible_challenger_evaluations'] == 0
assert registry['service_alignment']['champion_aligned'] is False
assert registry['current_decision']['action'] == 'retain_champion'
assert registry['current_decision']['production_deployment_approved'] is False
'Module 12 registry audit passed: retain TF-IDF champion'

'Module 12 registry audit passed: retain TF-IDF champion'

## Decision and next evidence

TF-IDF remains the champion. Frozen RoBERTa and revised LoRA remain challengers; full RoBERTa fine-tuning remains planned for real CUDA. The next valid promotion step is not another BANKING77 test run—it is acquiring, governing and locking a genuinely external evaluation dataset before any candidate sees its text or labels.